In [ ]:
%matplotlib inline
import sys
sys.path.append("../..")

import numpy as np
from matplotlib import pyplot as plt
from IPython.display import clear_output
import time
import datetime
from pypanodecoder import eventbuilder
from pypanodecoder import pedestals
import csv

In [ ]:
# Load the Crab run list from the CSV file
# Relevant columns: 
# 0 : UTC date
# 5 : UTC start time
# 6 : UTC stop time
# 8 : EAST/WEST flag (string)
with open("/data/CTA02/fegan/PANOSETI/CrabDataset.csv", newline="", encoding="utf-8") as file:
    all_crab = list(csv.reader(file))[1:]

In [ ]:
# Glob patterns to reach all Crab runs in the dataset
data = [ 
    '/data/CTA02/fegan/PANOSETI/data/L0/202601*/Fern/pcap/*onsky*pcapng',
    '/data/CTA02/fegan/PANOSETI/data/L0/202601*/Fern/pcap/rawdata/*onsky*pcapng' 
]

In [ ]:
# GTI run list for WEST and EAST runs, formatted as dictionaries with 'start' and 'stop' keys
crab_W = [ { 'start': f'{r[0]} {r[5]}', 'stop': f'{r[0]} {r[6]}' } for r in all_crab if r[8]=='WEST' ]
crab_E = [ { 'start': f'{r[0]} {r[5]}', 'stop': f'{r[0]} {r[6]}' } for r in all_crab if r[8]=='EAST' ]

In [ ]:
# Load all camera images from the dataset that fall within the GTIs for the WEST runs
# This may take a while to run, as it terates through all the specified pcapng files
images_W = eventbuilder.load_pcap_camera_images(data, gtis=crab_W)

In [ ]:
# Fit constant pedestal offset to the images (per GTI) and apply the correction
pedsub_images_W = pedestals.apply_polynomial_pedestal_correction(images_W, 0)

In [ ]:
for i in range(pedsub_images_W.images.shape[-1]):
    clear_output(wait=True)  # Clear previous output
    pc = plt.imshow(pedsub_images_W.images[...,i])
    plt.title(f'{str(datetime.datetime.fromtimestamp(np.round(pedsub_images_W.event_times[i]), tz=datetime.timezone.utc))[:-6]} UTC')
    plt.clim(0,300)
    plt.colorbar(pc)
    plt.show()
    time.sleep(0.05)
    plt.close()